<style>
.custom-spacing {
    line-height: 0.5;
}
</style>

### <center>**TUGAS INDIVIDU**</center>
### <center>**KECERDASAN ARTIFISIAL UNTUK GEOFISIKA**</center>
#### <center>**Dosen Pengampu:** 
#### <center>**Dr. Muhammad Alimuddin Hamzah, M. Eng**</center>
#### <center>**Prof.  Dr. Samsu Arif, M. Si**</center>
#### <center>**Muhammad Farid Wajedy, S.Si, M.Si**</center>

<div style="text-align:center;">
  <img src="https://www.unhas.ac.id/wp-content/uploads/2022/09/Logo-Resmi-Unhas-1.png" alt="Deskripsi Gambar" style="width:280px;">
</div>

### <center>**DISUSUN OLEH :**</center>

#### <center>**SITI SARAH CHAIRANI           (H061241022)**<center>

### <center>**PROGRAM STUDI GEOFISIKA**</center>
### <center>**FAKULTAS MATEMATIKA DAN ILMU PENGETAHUAN ALAM**</center>
#### <center>**UNIVERSITAS HASANUDDIN**</center>
#### <center>**MAKASSAR**</center>
#### <center>**2026**</center>

<hr>

# Form Prompt AI untuk Geofisika (Multi-Metode)

Struktur mengikuti Bab 2 *AI Cookbook untuk Earth Sciences* (prompt sebagai *work specification*), sesuai diagram pada Gambar 2.1.

<div align="center">

| No | Metode Geofisika | Topik Prompt |
|----|------------------|--------------|
| 1 | Geolistrik (VES) | Perhitungan resistivitas semu dan fitting model 1D tiga lapisan |
| 2 | Seismik refraksi | Estimasi kecepatan lapisan dan kedalaman bidang bias (metode intercept-time) |
| 3 | Gravitasi | Filter derivatif (FHD dan SVD) untuk delineasi struktur dari grid anomali Bouguer |
| 4 | Seismologi | Estimasi magnitude of completeness (Mc) dan b-value Gutenberg-Richter |
| 5 | Magnetotellurik (MT) | QC kurva sounding MT dan transformasi Niblett-Bostick |

</div>

<br>

<div align="center">

| Kode | Komponen | Fungsi |
|------|----------|--------|
| **R** | Role | Sudut pandang kerja, bukan credential |
| **T** | Task | Verb yang bisa diperiksa + objek + domain + kondisi penerimaan |
| **C** | Context | Tujuan, audiens, workflow, keputusan yang didukung |
| **I** | Input | Path, format, variabel, unit, CRS/datum, provenance |
| **K** | Constraints | File yang tidak boleh diubah, method, package, jaringan, approval |
| **O** | Output | Kontrak output: path, kolom/unit, format (rencana teks, belum hasil) |
| **Q** | Quality criteria | Kriteria yang dapat dinilai eksplisit |
| **V** | Optional verification | Check + expected + tindakan bila gagal |

</div>

> Bagian **O** adalah *expected output* berupa narasi rencana, bukan *actual result* dan bukan gambar.

## Template Umum

```markdown
## [R] Role
Bertindaklah sebagai <sudut pandang kerja>. Fokus pada <apa yang diperiksa>.

## [T] Task
<Verb> <objek> <domain/kondisi>, lalu <verb kedua>. <Kondisi penerimaan>.

## [C] Context
<Tujuan>, <audiens>, <tahap workflow>. Hasil dipakai untuk <keputusan>, bukan untuk <hal yang tidak dilakukan>.

## [I] Input
- `<path>`: format, variabel/kolom, unit, CRS/datum, sumber

## [K] Constraints
- Jangan mengubah <file/folder>.
- Method: <yang boleh>; <yang dilarang>.
- Software: <yang tersedia>; tanpa jaringan.
- Bila <kondisi tak jelas>: berhenti dan tanyakan.

## [O] Output (rencana teks)
- `<path output>`: <isi, kolom, unit>

## [Q] Quality criteria
- <kriteria terukur>

## [V] Optional verification
- Check: <command>. Expected: <kondisi>. Jika gagal: <tindakan>.
```

## Prompt 1 — Geolistrik: Resistivitas Semu dan Fitting Model 1D pada Data VES

### [R] Role
Bertindaklah sebagai reviewer interpretasi geolistrik 1D. Fokus pada ketepatan perhitungan resistivitas semu, kualitas kurva sounding, dan ketidakunikan (ekuivalensi) model lapisan.

### [T] Task
Hitung resistivitas semu (ρa) dari data Vertical Electrical Sounding konfigurasi Schlumberger `data/raw/ves_01.csv`, tandai titik yang menyimpang pada kurva ρa terhadap AB/2, lalu lakukan fitting model 1D dengan 3 lapisan dan laporkan resistivitas, ketebalan, serta misfit-nya. Jangan menafsirkan litologi atau potensi akuifer.

### [C] Context
Survei VES pada satu titik di dataran aluvial untuk investigasi awal air tanah. Hasil dipakai tim hidrogeologi untuk memutuskan titik mana yang perlu diverifikasi lebih lanjut. Hasil bukan penetapan lokasi atau kedalaman sumur bor.

### [I] Input
- `data/raw/ves_01.csv`: kolom `ab2_m` (m), `mn2_m` (m), `I_mA` (mA), `dV_mV` (mV). Bentangan AB/2 maksimum `{{m}}`. Sumber alat: `{{merek/model}}`.
- Faktor geometri Schlumberger: `K = π · ((AB/2)² − (MN/2)²) / MN`, dengan `MN = 2 · (MN/2)`. Rumus ini yang dipakai dan harus dicatat.
- Jumlah lapisan model: `3`. Rentang awal resistivitas: `{{min–max Ω·m}}`.

### [K] Constraints
- Jangan mengubah `data/raw/`; jangan menghapus baris data.
- Titik outlier hanya diberi flag dengan ambang `{{ambang}}` yang dinyatakan; jangan dibuang atau dihaluskan diam-diam.
- Bila terjadi lompatan ρa saat MN diganti (segmen kurva), laporkan; jangan menggeser (*shifting*) segmen tanpa persetujuan.
- Model tetap 3 lapisan; jangan menambah atau mengurangi lapisan sendiri.
- Software: Python dengan NumPy, SciPy, dan pandas; tanpa jaringan dan tanpa software inversi tertutup.
- Jika `mn2_m` lebih besar dari `ab2_m` atau satuan tidak jelas: berhenti dan tanyakan.

### [O] Output (rencana teks)
- `results/ves/ves_01_rhoa.csv`: `ab2_m`, `mn2_m`, `K_m`, `rho_a_ohm_m`, `flag`, `note`.
- `results/ves/ves_01_model.csv`: `layer`, `rho_ohm_m`, `thickness_m`, `depth_top_m`, `depth_bottom_m` (lapisan ke-3 dengan ketebalan tak hingga).
- `results/ves/ves_01_report.md`: narasi jumlah titik data, titik yang di-flag, hasil fitting, nilai RMS misfit (%), dan catatan ekuivalensi model.
- `results/ves/run_config.json`: rumus K, ambang flag, tebakan awal, algoritma optimasi, versi package.

### [Q] Quality criteria
- Jumlah baris `ves_01_rhoa.csv` sama dengan jumlah baris input.
- Setiap flag memuat nilai, ambang, dan alasan.
- Satuan (m, Ω·m) eksplisit pada semua kolom.
- Laporan memisahkan fakta terukur, asumsi model, dan hal yang belum bisa ditafsirkan.
- Nilai misfit dilaporkan bersama pernyataan bahwa lebih dari satu model dapat cocok dengan data.

### [V] Optional verification
- Check: buat data sintetis dari model 3 lapisan yang diketahui, lalu jalankan prosedur yang sama. Expected: parameter model terpulihkan dalam `{{toleransi}}`%. Jika gagal: laporkan bug pada forward model atau optimasi.
- Check: hitung ulang `rho_a` untuk 5 baris acak dengan script terpisah. Expected: sama dengan `ves_01_rhoa.csv`. Jika gagal: periksa rumus K dan satuan MN.
- Check: SHA-256 `ves_01.csv` sebelum dan sesudah. Expected: identik. Jika gagal: hentikan dan laporkan.

## Prompt 2 — Seismik Refraksi: Kecepatan Lapisan dan Kedalaman Bidang Bias (Intercept-Time)

### [R] Role
Bertindaklah sebagai reviewer interpretasi seismik refraksi dangkal. Fokus pada pemisahan segmen gelombang langsung dan gelombang bias, konsistensi kecepatan forward–reverse, dan asumsi geometri lapisan.

### [T] Task
Dari picks first-break `data/raw/refraction_picks.csv` (shot forward dan reverse), tentukan segmen gelombang langsung dan gelombang bias dengan regresi linear terpisah, hitung V1, V2, intercept time, dan crossover distance, lalu hitung kedalaman bidang bias h1 dengan metode intercept-time untuk kedua arah shot. Jangan menjalankan tomografi atau inversi.

### [C] Context
Survei refraksi dangkal untuk memperkirakan kedalaman batuan dasar pada lokasi rencana bangunan. Hasil dipakai sebagai estimasi awal oleh geoteknik. Hasil tidak dipakai untuk menetapkan kapasitas dukung tanah atau desain pondasi.

### [I] Input
- `data/raw/refraction_picks.csv`: `shot_id`, `shot_x_m` (m), `receiver_x_m` (m), `t_fb_ms` (ms), `direction` (`forward`/`reverse`).
- Spasi geophone: `{{m}}`; jumlah geophone: `{{n}}`.
- Model: 2 lapisan, bidang bias dianggap datar. Rumus: `h1 = (t_i · V1 · V2) / (2 · √(V2² − V1²))`, dengan `t_i` intercept time.
- Titik pemisah segmen (breakpoint): `{{manual: offset m / otomatis: kriteria}}`.

### [K] Constraints
- Jangan mengubah file picks; jangan melakukan picking ulang dari trace.
- Breakpoint harus dinyatakan (manual atau kriteria otomatis) dan dicatat; jangan dipilih diam-diam.
- Model tetap 2 lapisan; jangan menambah lapisan ketiga.
- Jika V2 forward dan V2 reverse berbeda lebih dari `{{ambang}}`%, beri flag "indikasi bidang miring atau heterogen" dan jangan merata-ratakan tanpa keterangan.
- Software: Python, NumPy, pandas; tanpa jaringan.
- Jika V2 ≤ V1: berhenti dan laporkan (rumus tidak valid).

### [O] Output (rencana teks)
- `results/refraction/velocity_fit.csv`: `direction`, `V1_m_s`, `V2_m_s`, `intercept_ms`, `crossover_m`, `r2_direct`, `r2_refracted`, `n_direct`, `n_refracted`.
- `results/refraction/depth_estimate.csv`: `direction`, `h1_m`, `V_ratio_V2_V1`, `flag`.
- `results/refraction/report.md`: narasi pemilihan breakpoint, hasil V1, V2, dan h1 tiap arah, perbedaan forward–reverse, serta keterbatasan asumsi bidang datar.
- `results/refraction/run_config.json`: rumus, breakpoint, versi package.

### [Q] Quality criteria
- Setiap kecepatan dilaporkan bersama R² dan jumlah titik yang dipakai.
- Satuan m/s, ms, dan m eksplisit pada semua kolom.
- Perbedaan h1 forward–reverse dilaporkan, tidak disembunyikan.
- Picks yang tidak dipakai pada regresi tetap tercatat beserta alasannya.

### [V] Optional verification
- Check: buat picks sintetis dari model 2 lapisan (V1 = 400 m/s, V2 = 1800 m/s, h1 = 5 m) dan jalankan prosedur. Expected: V1, V2, h1 terpulihkan dalam `{{toleransi}}`%. Jika gagal: laporkan bug.
- Check: bandingkan crossover distance terhitung dengan pendekatan `x_c = 2 · h1 · √((V2 + V1)/(V2 − V1))`. Expected: selisih kecil. Jika besar: periksa breakpoint.
- Check: SHA-256 file input sebelum dan sesudah. Expected: identik.

## Prompt 3 — Gravitasi: Filter Derivatif (FHD dan SVD) untuk Delineasi Struktur

### [R] Role
Bertindaklah sebagai reviewer implementasi numerik untuk pengolahan data potensial. Fokus pada satuan turunan, efek tepi grid, dan pemisahan antara hasil filter dan interpretasi geologi.

### [T] Task
Hitung First Horizontal Derivative (FHD) dan Second Vertical Derivative (SVD) dari grid anomali Bouguer `data/processed/bouguer_grid.tif`, lalu ekstrak titik kandidat tepi anomali dari maksimum FHD dan kontur nol SVD. Tuliskan hasil sebagai tabel kandidat, bukan sebagai peta patahan.

### [C] Context
Survei gravitasi regional untuk pemetaan awal batas cekungan dan struktur. Hasil filter dipakai tim interpretasi sebagai bahan diskusi awal tentang kemenerusan lineament. Hasil tidak dipakai untuk menetapkan patahan, estimasi kedalaman, atau pemodelan inversi.

### [I] Input
- `data/processed/bouguer_grid.tif`: GeoTIFF, anomali Bouguer dalam mGal, CRS `{{EPSG:... UTM}}` (satuan meter), ukuran sel `{{m}}`, nodata `{{-9999}}`.
- Rumus: `FHD = √((∂g/∂x)² + (∂g/∂y)²)`; `SVD = −(∂²g/∂x² + ∂²g/∂y²)` (asumsi medan memenuhi persamaan Laplace).
- Metode turunan: `{{beda hingga / FFT}}`.

### [K] Constraints
- Jangan mengubah file grid asli.
- Jangan melakukan upward continuation, penghalusan, atau pemisahan regional–residual, kecuali dinyatakan pada Input.
- Sel nodata harus tetap nodata pada output; jangan diisi interpolasi.
- Jika CRS bukan sistem koordinat metrik, berhenti dan tanyakan (satuan turunan menjadi tidak valid).
- Tepi grid (`{{n}}` sel) diberi flag efek tepi.
- Jangan menyimpulkan keberadaan patahan atau kedalaman sumber.
- Software: Python dengan rasterio, NumPy, SciPy; tanpa jaringan.

### [O] Output (rencana teks)
- `results/gravity/fhd_mgal_per_m.tif` dan `results/gravity/svd_mgal_per_m2.tif`: grid dengan CRS, transform, dan nodata identik dengan input.
- `results/gravity/edge_candidates.csv`: `id`, `x_m`, `y_m`, `fhd_value`, `svd_value`, `edge_flag`.
- `results/gravity/report.md`: narasi statistik FHD dan SVD, jumlah kandidat, sel yang di-flag, dan pernyataan bahwa hasil belum diinterpretasi secara geologi.
- `results/gravity/run_config.json`: rumus, metode turunan, ukuran sel, versi package.

### [Q] Quality criteria
- Shape, CRS, dan transform output sama dengan input.
- Satuan mGal/m dan mGal/m² eksplisit.
- Sel nodata input tetap nodata pada output.
- Kriteria pemilihan kandidat (ambang FHD, toleransi SVD ≈ 0) tercatat.

### [V] Optional verification
- Check: ramp linear sintetis `g = a·x`. Expected: FHD konstan (= `a`) dan SVD ≈ 0. Jika gagal: laporkan bug pada perhitungan turunan.
- Check: grid sintetis berupa anomali bola/silinder. Expected: maksimum FHD berada di sekitar tepi anomali. Jika tidak: periksa ukuran sel dan orientasi sumbu.
- Check: SHA-256 file input sebelum dan sesudah. Expected: identik.

## Prompt 4 — Seismologi: Magnitude of Completeness (Mc) dan b-value Gutenberg-Richter

### [R] Role
Bertindaklah sebagai reviewer analisis statistik seismisitas. Fokus pada konsistensi tipe magnitudo, pemilihan Mc, dan ketidakpastian estimasi b-value.

### [T] Task
Dari katalog gempa yang telah dinormalisasi `data/processed/events.csv`, hitung frequency–magnitude distribution (FMD), tentukan Mc dengan metode maximum curvature, lalu estimasi b-value dengan maximum likelihood (Aki–Utsu) beserta ketidakpastiannya untuk event dengan M ≥ Mc. Jangan menyimpulkan tingkat bahaya gempa.

### [C] Context
Analisis awal seismisitas satu wilayah tektonik untuk eksplorasi data. Hasil dipakai sebagai gambaran kelengkapan katalog dan karakter statistik awal. Bukan seismic-hazard assessment dan bukan prediksi gempa.

### [I] Input
- `data/processed/events.csv`: `event_id`, `origin_time_utc`, `latitude`, `longitude`, `depth_km`, `magnitude`, `magnitude_type`, `agency`.
- Tipe magnitudo yang dipakai: `{{ML/mb/Mw}}` (hanya satu tipe). Bin magnitudo ΔM = `{{0.1}}`.
- Rumus: `b = log10(e) / (mean(M) − (Mc − ΔM/2))`; `σ_b = 2.30 · b² · √(Σ(Mi − mean(M))² / (N(N−1)))`.
- Batas minimum jumlah event untuk estimasi: `{{50}}`.

### [K] Constraints
- Jangan mengubah `data/processed/events.csv`.
- Jangan mencampur atau mengonversi tipe magnitudo; event dengan tipe lain dikecualikan dan dicatat jumlahnya.
- Jangan melakukan declustering kecuali dinyatakan pada Input.
- Jika jumlah event di atas Mc kurang dari batas minimum, jangan menghitung b-value: laporkan dan berhenti.
- Jangan menafsirkan nilai b terhadap tegangan, fluida, atau hazard.
- Software: Python, NumPy, pandas; tanpa jaringan.

### [O] Output (rencana teks)
- `results/seismicity/fmd.csv`: `mag_bin`, `n_incremental`, `n_cumulative`.
- `results/seismicity/bvalue_summary.json`: `mc`, `b_value`, `sigma_b`, `n_above_mc`, `magnitude_type`, `time_range`.
- `results/seismicity/report.md`: narasi jumlah event total dan yang dikecualikan, Mc, b-value dengan ketidakpastian, serta catatan keterbatasan (maximum curvature cenderung mengestimasi Mc terlalu rendah).
- `results/seismicity/run_config.json`: ΔM, metode Mc, rumus, versi package.

### [Q] Quality criteria
- Jumlah event yang dipakai + dikecualikan = jumlah baris input.
- b-value selalu dilaporkan bersama σ_b, Mc, dan N.
- Tipe magnitudo dan periode waktu dinyatakan eksplisit.
- Tidak ada interpretasi tektonik atau hazard pada laporan.

### [V] Optional verification
- Check: katalog sintetis dengan b = 1,0 (distribusi eksponensial magnitudo, Mc = 2,0, N = 1000). Expected: b terpulihkan dalam ± `{{0,1}}`. Jika gagal: laporkan bug.
- Check: bandingkan Mc pada ΔM = 0,1 dan 0,2. Expected: selisih dilaporkan sebagai sensitivitas. Jika besar: tandai Mc tidak stabil.
- Check: SHA-256 file input sebelum dan sesudah. Expected: identik.

## Prompt 5 — Magnetotellurik: QC Kurva Sounding dan Transformasi Niblett-Bostick

### [R] Role
Bertindaklah sebagai reviewer QC data magnetotellurik. Fokus pada kualitas per frekuensi, konsistensi resistivitas semu–fase, dan pemisahan transformasi awal dari inversi.

### [T] Task
Periksa kurva sounding MT `data/raw/mt_site01.csv` untuk menandai frekuensi dengan error besar, fase di luar rentang yang diharapkan, dan lompatan resistivitas semu antarfrekuensi, lalu hitung transformasi Niblett–Bostick (kedalaman dan resistivitas Bostick) untuk mode XY dan YX. Jangan menjalankan inversi 1D atau 2D.

### [C] Context
Survei MT untuk investigasi struktur resistivitas dalam pada area prospek geotermal. QC dilakukan sebelum data diserahkan ke tahap inversi. Hasil transformasi Bostick hanya gambaran awal, bukan model resistivitas final dan bukan interpretasi reservoir.

### [I] Input
- `data/raw/mt_site01.csv`: `freq_hz` (Hz), `rho_xy`, `rho_yx` (Ω·m), `phi_xy`, `phi_yx` (derajat), `rho_err_pct`, `phi_err_deg`. Konvensi fase: `{{semua fase di kuadran 0–90° / YX di kuadran ketiga}}`.
- Rumus: `D_B = √(ρa / (ω · μ0))`, dengan `ω = 2π·f`, `μ0 = 4π × 10⁻⁷ H/m`; `ρ_B = ρa · (π / (2·φ) − 1)`, dengan φ dalam radian.
- Ambang: `rho_err_pct > {{10}}`, `phi_err_deg > {{5}}`, lompatan ρa antarfrekuensi bersebelahan > `{{faktor}}`.

### [K] Constraints
- Jangan mengubah `data/raw/`; jangan menghapus frekuensi, hanya beri flag.
- Jangan menjalankan inversi, static-shift correction, atau smoothing.
- Konvensi fase harus dinyatakan; jika tidak jelas, berhenti dan tanyakan (bisa membuat ρ_B salah).
- Bila φ ≥ 90° atau φ ≤ 0°, transformasi Bostick tidak dihitung untuk frekuensi tersebut dan alasannya dicatat.
- Software: Python dengan NumPy dan pandas; tanpa jaringan.
- Jangan mengaitkan hasil dengan reservoir, zona clay cap, atau sumber panas.

### [O] Output (rencana teks)
- `results/mt/mt_site01_qc.csv`: seluruh kolom input + `qc_flag` + `qc_note`.
- `results/mt/mt_site01_bostick.csv`: `freq_hz`, `mode`, `depth_bostick_m`, `rho_bostick_ohm_m`, `status`.
- `results/mt/qc_report.md`: narasi jumlah frekuensi, jumlah flag per jenis, rentang frekuensi yang andal, dan frekuensi yang tidak dapat ditransformasi.
- `results/mt/run_config.json`: ambang, konvensi fase, konstanta, versi package.

### [Q] Quality criteria
- Jumlah baris `qc.csv` sama dengan jumlah baris input; urutan frekuensi tidak berubah.
- Semua ambang dan konvensi fase tercatat.
- Satuan Hz, Ω·m, derajat, dan m eksplisit pada semua kolom.
- Laporan memisahkan hasil terukur, asumsi, dan hal yang belum ditafsirkan.

### [V] Optional verification
- Check: setengah-ruang homogen sintetis (ρ = 100 Ω·m, φ = 45°). Expected: ρ_B ≈ 100 Ω·m di semua frekuensi dan `D_B` sesuai rumus. Jika gagal: periksa satuan fase dan konstanta.
- Check: hitung ulang `D_B` untuk 5 frekuensi acak secara independen. Expected: sama dengan `bostick.csv`. Jika gagal: periksa penggunaan ω dan μ0.
- Check: SHA-256 file input sebelum dan sesudah. Expected: identik.

## Cara Pakai
1. Pilih satu prompt dan isi semua `{{...}}` serta path file.
2. Jalankan, lalu nilai hasil aktual terhadap **[Q]** dan **[V]**.
3. Bila prompt terlalu panjang, pindahkan detail ke file configuration/recipe pendamping dan pertahankan **Constraints** serta **Quality criteria** di prompt.
4. Simpan diff dan `run_config.json` sebagai bukti perubahan.